# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zoha200/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Lane 2: Refresh / Content Opportunity Scoring.

This is a SCORING task (a form of ranking): for every page, I want a score that
represents "how urgently does this page need review," so an editor can sort their
whole backlog by that score and work top-down. It's not classification in the pure
sense — the deliverable isn't a single yes/no per page, it's a rank-ordered list,
because the editor's real constraint is limited time, not a binary decision per page.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zoha200/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows loaded")


30000 rows loaded


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target/proxy: whether a page is declining, defined as impressions in the most recent
30-day window being lower than the prior 30-day window (impressions_last_30d
impressions_prev_30d).

Where this comes from: this is an OBSERVED outcome, built directly from two real,
disjoint traffic windows already in the data — not a defined business rule and not
the pre-baked trend_direction/trend_pct columns ML-02 flagged as current-state labels
too close to leakage. My proxy agrees with trend_direction=="down" 88.5% of the time,
which tells me they're capturing a similar real pattern, but my version is built from
raw numbers I can defend, not an opaque bucket.

For the actual capstone, this proxy should shift further forward — comparing a
FUTURE 30-day window against today, once the label is genuinely about what happens
next rather than what already happened.

In [2]:
df["is_declining_proxy"] = (df["impressions_last_30d"] < df["impressions_prev_30d"]).astype(int)

print(f"Declining by 30d-window proxy: {df['is_declining_proxy'].sum()} ({100*df['is_declining_proxy'].mean():.1f}% of pages)")

agree = (df["is_declining_proxy"] == (df["trend_direction"]=="down").astype(int)).mean()
print(f"Agreement with trend_direction=='down' bucket: {100*agree:.1f}%")


Declining by 30d-window proxy: 19716 (65.7% of pages)
Agreement with trend_direction=='down' bucket: 88.5%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Precision@50 — of the top 50 pages the model flags as declining,
what fraction actually are declining (by my proxy label).

Why this one: an editor only has time to review a fixed number of pages per week,
not the whole backlog. What matters is that the TOP of the ranked list is trustworthy
— if 40 of the top 50 are real, the editor's time is well spent; if only 15 are real,
they lose trust in the tool and stop using it. Precision@50 measures exactly that,
rather than a metric like overall accuracy, which would reward the model for being
right about the thousands of obviously-fine pages nobody needed flagged in the first
place.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("precision_at_k defined — will be used once a model/score exists to evaluate")

precision_at_k defined — will be used once a model/score exists to evaluate


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one row = one content page (content_id), belonging to one client
(client_id). This matches the actual decision being supported — an editor reviews
one page at a time, not a whole client account at once.

In [4]:
cols = ["content_id", "client_id", "content_type", "days_since_last_update",
        "impressions_prev_30d", "impressions_last_30d", "is_declining_proxy"]
print(df[cols].head(5).to_string(index=False))


          content_id         client_id    content_type  days_since_last_update  impressions_prev_30d  impressions_last_30d  is_declining_proxy
content_304f48230142 client_f369cb89fc keyword article                      20                   987                   578                   1
content_a1fb4e703a9e client_4e07408562 keyword article                      25                  5915                  2501                   1
content_9aa793d4d895 client_7f2253d7e2 keyword article                      20                  6089                  2382                   1
content_331d6c4de07b client_19581e27de keyword article                      22                  4206                  3626                   1
content_d99b7a2d90ca client_3fdba35f04 keyword article                      14                  6452                  4211                   1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "flag pages that are stale AND still getting traffic" sounds
reasonable, but in the ML-01 data, only 0.5% of actually-declining pages met both
conditions at once — meaning a simple staleness rule would miss almost all real
decline. The pattern that actually predicts decline involves several signals
interacting (age, position, CTR, traffic trend) in ways that aren't expressible as
one or two clean thresholds. That's exactly the kind of messy, multi-signal pattern
ML is suited for: in ML-01, a random forest scored 0.740 Precision@50 versus a
hand-written rule's 0.240 on a related question — roughly 3x more of its top-50 picks
were correct, direct evidence the pattern isn't rule-shaped.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.